In [1]:
%pip install qiskit qiskit-aer qiskit-ibm-runtime matplotlib pylatexenc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.7/120.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.3/113.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.2/234.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/13

In [1]:
import math
from fractions import Fraction
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFT
from qiskit.visualization import plot_histogram

def c_amod15(power):
    """
    Creates a controlled unitary gate that multiplies the target by 2^power mod 15.
    Since 15 is 1111 in binary, multiplying by 2 mod 15 is effectively a
    circular left shift of the 4 qubits.
    """
    U = QuantumCircuit(4)
    for _ in range(power):
        U.swap(0, 3)
        U.swap(0, 2)
        U.swap(0, 1)

    U_gate = U.to_gate(label=f"2^{power} mod 15")
    return U_gate.control(1)

# 1. Define Register Sizes
N_COUNT = 8  # Evaluation qubits (dictates precision)
TARGET_Q = 4 # Target qubits (needs enough bits to represent 15)

qc = QuantumCircuit(N_COUNT + TARGET_Q, N_COUNT)

# 2. Initialization
# Apply Hadamard gates to the evaluation register to create superposition
for q in range(N_COUNT):
    qc.h(q)

# Initialize the target register to the state |1> (computational basis)
qc.x(N_COUNT)

# 3. Apply the Controlled-U Operations (The Oracle)
# We apply 2^(2^j) mod 15 for each qubit in the evaluation register
for q in range(N_COUNT):
    power = 2**q
    qc.append(c_amod15(power), [q] + [i + N_COUNT for i in range(TARGET_Q)])

# 4. Apply the Inverse Quantum Fourier Transform (QFT†)
# This extracts the phase from the Fourier basis to the computational basis
qc.append(QFT(N_COUNT, inverse=True).to_instruction(), range(N_COUNT))

# 5. Measurement
# Measure only the evaluation register
qc.measure(range(N_COUNT), range(N_COUNT))

# 6. Execution on a Simulator
simulator = AerSimulator()
compiled_circuit = transpile(qc, simulator)
job = simulator.run(compiled_circuit, shots=1024)
result = job.result()
counts = result.get_counts()

# 7. Classical Post-Processing (Extracting the Factors)
print("--- Quantum Measurement Results ---")
print(f"Raw Measurement Counts: {counts}\n")

# Find the most frequent measurement (ignoring the 0 case which gives no period)
filtered_counts = {k: v for k, v in counts.items() if int(k, 2) != 0}
measured_str = max(filtered_counts, key=filtered_counts.get)
measured_int = int(measured_str, 2)

# Calculate the phase (theta)
phase = measured_int / (2**N_COUNT)
print(f"Most frequent binary measurement: {measured_str} (Integer: {measured_int})")
print(f"Calculated Phase (θ): {measured_int} / 256 = {phase}")

# Use continued fractions to find the period (r)
fraction = Fraction(phase).limit_denominator(15)
r = fraction.denominator
print(f"Extracted Period (r) via continued fractions: {r}\n")

# 8. Classical Factoring via GCD
if r % 2 != 0:
    print("Failed: Period is odd. In a real scenario, choose a new 'a'.")
else:
    a = 2
    guess1 = math.gcd(a**(r//2) - 1, 15)
    guess2 = math.gcd(a**(r//2) + 1, 15)

    print("--- Final Factorization ---")
    print(f"Factor 1: {guess1}")
    print(f"Factor 2: {guess2}")
    if guess1 * guess2 == 15:
        print("Success: 15 has been factorized.")

/tmp/ipykernel_2446/4280924005.py:45: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(N_COUNT, inverse=True).to_instruction(), range(N_COUNT))


--- Quantum Measurement Results ---
Raw Measurement Counts: {'00000000': 218, '01000000': 263, '10000000': 265, '11000000': 278}

Most frequent binary measurement: 11000000 (Integer: 192)
Calculated Phase (θ): 192 / 256 = 0.75
Extracted Period (r) via continued fractions: 4

--- Final Factorization ---
Factor 1: 3
Factor 2: 5
Success! 15 has been factorized.
